In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("cleaned_data.csv")
data.shape,data.columns

((99264, 3), Index(['review', 'label', 'source'], dtype='object'))

In [3]:
reviews = data.iloc[1300:2000]
reviews.shape

(700, 3)

In [4]:
import json
import os
import re
import torch
from tqdm import tqdm
from transformers import pipeline
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM

In [5]:
import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv('HF_TOKENS.env')
active_token = os.getenv("HF_TOKEN1")

if active_token:
    login(token=active_token)
    print("Logged in successfully with the selected token.")
else:
    print("Requested token not found in .env file.")

Logged in successfully with the selected token.


In [6]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
OUTPUT_FILE = "absa_checkpoints2.jsonl"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
system_prompt = (
    "You are an expert NLP annotator for Hinglish and English reviews. "
    "Extract all aspect terms and their specific sentiment (+1 for positive, -1 for negative). "
    "Output ONLY a JSON list of objects with 'aspect' and 'sentiment' keys. "
    "Example: [{'aspect': 'battery', 'sentiment': 1}, {'aspect': 'camera quality', 'sentiment': -1}]"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto"
)

# generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
def absa_with_checkpoints(df, output_path, b_size):
    done_ids = set()
    if os.path.exists(output_path):
        with open(output_path, 'r') as f:
            for line in f:
                try:
                    done_ids.add(json.loads(line)['id'])
                except:
                    continue

    todo_df = df[~df.index.isin(done_ids)]
    print(f"Already processed: {len(done_ids)}. Remaining: {len(todo_df)}")

    model.eval()

    with open(output_path, 'a', encoding='utf-8') as f:
        for i in tqdm(range(0, len(todo_df), b_size)):
            batch = todo_df.iloc[i : i + b_size]
            prompts = []

            for _, row in batch.iterrows():
                messages = [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Review: {row['review']}"}
                ]
                text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True
                )
                prompts.append(text)

            try:
                inputs = tokenizer(
                    prompts,
                    return_tensors="pt",
                    padding=True,
                    truncation=True
                ).to(model.device)

                with torch.inference_mode():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=156,
                        temperature=0.1,
                        do_sample=False
                    )

                decoded = tokenizer.batch_decode(
                    outputs,
                    skip_special_tokens=True
                )

                for idx, (original_idx, row) in enumerate(batch.iterrows()):
                    raw_text = decoded[idx]

                    match = re.search(r'\[.*\]', raw_text, re.DOTALL)
                    clean_json_str = match.group() if match else "[]"

                    try:
                        parsed_annotations = json.loads(clean_json_str)
                    except:
                        repaired = clean_json_str.replace("'", '"')
                        try:
                            parsed_annotations = json.loads(repaired)
                        except:
                            parsed_annotations = []

                    result = {
                        "id": int(original_idx),
                        "review": row['review'],
                        "annotations": parsed_annotations
                    }

                    f.write(json.dumps(result) + "\n")

                f.flush()

            except Exception as e:
                print(f"Critical error in batch starting at {i}: {e}")
                continue

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
absa_with_checkpoints(reviews,OUTPUT_FILE,b_size=8)

Already processed: 376. Remaining: 692



  0%|          | 0/87 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
  3%|▎         | 3/87 [05:53<2:42:21, 115.97s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
